## Project Summary

This BaristaBot implementation demonstrates key LangGraph concepts:

### Key Features Implemented

1. **State Management with TypedDict**
   - OrderState manages conversation history, order items, and completion status
   - add_messages annotation enables message appending (not replacement)

2. **Node Functions**
   - `chatbot_with_tools`: LLM-based conversation node with tool support
   - `human_node`: User input and interaction handling
   - `order_node`: State manipulation for order management
   - `tool_node`: Automated menu lookup

3. **Conditional Routing**
   - `maybe_route_to_tools`: Routes between auto-tools, order tools, human, and exit
   - `maybe_exit_human_node`: Allows user to quit or continue

4. **Tool Integration**
   - `get_menu`: Provides dynamic menu access
   - `add_to_order`, `confirm_order`, `get_order`, `clear_order`, `place_order`: Order management
   - Tools are bound to the LLM for automatic tool selection

5. **Conversation Flow**
   - Welcome message initiates the conversation
   - Natural language processing for order understanding
   - Menu lookup before adding items
   - Order confirmation before placement
   - Graceful exit handling

### What This Demonstrates

- How to structure a real-world application with LangGraph
- State management across multiple nodes
- Tool-augmented LLM applications
- Conditional branching and looping
- Integration with Gemini API via LangChain

### Things to Try

1. Order a simple drink: "I'd like a latte"
2. Ask about menu: "What teas do you have?"
3. Make modifications: "Can I get that with oat milk?"
4. Confirm and place: Follow the bot's confirmation request
5. Exit gracefully: Type 'q' or 'quit'

In [ ]:
print("=" * 70)
print("STARTING BARISTABOT - Interactive Cafe Ordering System")
print("=" * 70)
print("\nInstructions:")
print("- Type your order requests in natural language")
print("- Ask 'menu' or 'what do you have' to see available items")
print("- Type 'q', 'quit', 'exit', or 'goodbye' to leave")
print("=" * 70)
print()

# The default recursion limit for traversing nodes is 25
# Setting it higher means you can have a more complex order with multiple steps
config = {"recursion_limit": 100}

try:
    # Run the complete ordering system
    state = graph_with_order_tools.invoke({"messages": [], "order": [], "finished": False}, config)
    
    print("\n" + "=" * 70)
    print("Order Complete! Thank you for using BaristaBot")
    print("=" * 70)
    
except KeyboardInterrupt:
    print("\n\nSession interrupted by user.")
except Exception as e:
    print(f"\nAn error occurred: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Visualize the graph
try:
    graph_image = graph_with_order_tools.get_graph().draw_mermaid_png()
    Image(graph_image)
except Exception as e:
    print(f"Could not render graph image: {e}")
    print("Graph is still functional, just cannot display image")

In [ ]:
# Set up tools and the complete graph

# Auto-tools will be invoked automatically by the ToolNode
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

# Order-tools will be handled by the order node
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

# The LLM needs to know about all of the tools
llm_with_tools = llm.bind_tools(auto_tools + order_tools)

# Create the graph
graph_builder = StateGraph(OrderState)

# Add the nodes
graph_builder.add_node("chatbot", lambda state: chatbot_with_welcome_msg(state) | {"order": state.get("order", []), "finished": state.get("finished", False)})
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Update chatbot to use tools
def chatbot_with_tools(state: OrderState) -> OrderState:
    """The chatbot with tools. A wrapper around the model's chat interface."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    return defaults | state | {"messages": [new_output]}

# Clear the previous graph builder and recreate
graph_builder = StateGraph(OrderState)

# Add all nodes
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Set up edges
graph_builder.add_edge(START, "chatbot")

# Chatbot -> {tools, ordering, human, END}
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)

# Human -> {chatbot, END}
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools always route back to chat
graph_builder.add_edge("tools", "chatbot")

# Ordering always routes back to chat
graph_builder.add_edge("ordering", "chatbot")

# Compile the graph
graph_with_order_tools = graph_builder.compile()

print("Complete graph built successfully!")
print("\nGraph structure:")
print(graph_with_order_tools.get_graph())

In [ ]:
def maybe_route_to_tools(state: OrderState) -> Literal["tools", "ordering", "human"]:
    """Route between human, tool nodes, and ordering node based on LLM output."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    if state.get("finished", False):
        # When an order is placed, exit the app
        return END

    elif hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        # Check if any tool calls are for auto-tools (menu)
        auto_tool_names = {"get_menu"}
        order_tool_names = {"add_to_order", "confirm_order", "get_order", "clear_order", "place_order"}
        
        tool_call_names = {tool["name"] for tool in msg.tool_calls}
        
        if tool_call_names & auto_tool_names:
            return "tools"
        elif tool_call_names & order_tool_names:
            return "ordering"
        else:
            return "human"
    else:
        return "human"

print("Routing functions defined!")

In [ ]:
def order_node(state: OrderState) -> OrderState:
    """The ordering node. This is where the order state is manipulated."""
    # Get the last message which should contain tool calls
    msgs = state.get("messages", [])
    tool_msg = msgs[-1]
    
    order = state.get("order", [])
    outbound_msgs = []
    order_placed = False

    if not hasattr(tool_msg, "tool_calls"):
        return {"messages": [], "order": order, "finished": False}

    for tool_call in tool_msg.tool_calls:

        if tool_call["name"] == "add_to_order":
            # Each order item is assembled as "drink (modifiers, ...)"
            drink = tool_call["args"]["drink"]
            modifiers = tool_call["args"].get("modifiers", [])
            
            if isinstance(modifiers, str):
                modifiers = [modifiers]
            else:
                modifiers = list(modifiers) if modifiers else []
            
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"
            order.append(f'{drink} ({modifier_str})')
            response = "\n".join(order)

        elif tool_call["name"] == "confirm_order":
            # Display the order to the user and wait for confirmation
            print("\nYour order:")
            if not order:
                print("  (no items)")
            else:
                for drink in order:
                    print(f"  {drink}")

            response = input("Is this correct? ")

        elif tool_call["name"] == "get_order":
            response = "\n".join(order) if order else "(no order)"

        elif tool_call["name"] == "clear_order":
            order.clear()
            response = "Order cleared."

        elif tool_call["name"] == "place_order":
            order_text = "\n".join(order)
            print("\nSending order to kitchen!")
            print("Order:")
            for item in order:
                print(f"  {item}")

            order_placed = True
            response = str(randint(1, 5))  # ETA in minutes

        else:
            raise NotImplementedError(f'Unknown tool call: {tool_call["name"]}')

        # Record the tool results as tool messages
        outbound_msgs.append(
            ToolMessage(
                content=str(response),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": outbound_msgs, "order": order, "finished": order_placed}

print("Order node function defined!")

In [ ]:
def human_node(state: OrderState) -> OrderState:
    """Display the last model message to the user, and receive the user's input."""
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)

    user_input = input("User: ")

    # If it looks like the user is trying to quit, flag the conversation as over
    if user_input in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True

    return state | {"messages": [HumanMessage(content=user_input)]}


def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    """The chatbot with welcome message support."""
    
    if state["messages"]:
        # If there are messages, continue the conversation with the Gemini model
        new_output = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        # If there are no messages, start with the welcome message
        new_output = AIMessage(content=WELCOME_MSG)

    return state | {"messages": [new_output]}


def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", END]:
    """Route to the chatbot, unless it looks like the user is exiting."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"

print("Human node functions defined!")

In [ ]:
# Order manipulation tools - these are stubs that will be implemented in order_node
@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers.

    Args:
        drink: The name of the drink to add
        modifiers: List of modifiers to apply to the drink

    Returns:
      The updated order in progress.
    """
    pass


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct.

    Returns:
      The user's free-text response.
    """
    pass


@tool
def get_order() -> str:
    """Returns the users order so far. One item per line."""
    pass


@tool
def clear_order() -> str:
    """Removes all items from the user's order.
    
    Returns:
      Confirmation message.
    """
    pass


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment.

    Returns:
      The estimated number of minutes until the order is ready.
    """
    pass

print("Order manipulation tools defined!")

In [ ]:
@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return """
    MENU:
    Coffee Drinks:
    - Espresso
    - Americano
    - Cold Brew

    Coffee Drinks with Milk:
    - Latte
    - Cappuccino
    - Cortado
    - Macchiato
    - Mocha
    - Flat White

    Tea Drinks:
    - English Breakfast Tea
    - Green Tea
    - Earl Grey

    Tea Drinks with Milk:
    - Chai Latte
    - Matcha Latte
    - London Fog

    Other Drinks:
    - Steamer
    - Hot Chocolate

    Modifiers:
    - Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default: Whole
    - Espresso shots: Single, Double, Triple, Quadruple; Default: Double
    - Caffeine: Decaf, Regular; Default: Regular
    - Hot-Iced: Hot, Iced; Default: Hot
    - Sweeteners (add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    - Special requests: any reasonable modification (e.g., 'extra hot', 'one pump', 'half caff', 'extra foam')
    
    Notes:
    - "dirty" means add espresso to a drink that doesn't usually have it (e.g., "Dirty Chai Latte")
    - "Regular milk" is the same as "Whole milk"
    - "Sweetened" means add regular sugar, not a sweetener
    - Soy milk is out of stock today
  """

print("Menu tool defined!")

In [ ]:
def chatbot(state: OrderState) -> OrderState:
    """The basic chatbot node that invokes the LLM."""
    message_history = [BARISTABOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}


# Set up the initial graph based on our state definition
graph_builder = StateGraph(OrderState)

# Add the chatbot function
graph_builder.add_node("chatbot", chatbot)

# Define the chatbot node as the app entrypoint
graph_builder.add_edge(START, "chatbot")

# We'll compile just to show the structure for now
chat_graph = graph_builder.compile()

print("Basic graph created. Structure:")
print(chat_graph.get_graph())

In [ ]:
class OrderState(TypedDict):
    """State representing the customer's order conversation."""
    
    # The chat conversation. This preserves the conversation history
    # between nodes. The `add_messages` annotation indicates to LangGraph
    # that state is updated by appending returned messages, not replacing them.
    messages: Annotated[list, add_messages]
    
    # The customer's in-progress order.
    order: list[str]
    
    # Flag indicating that the order is placed and completed.
    finished: bool


# The system instruction defines how the chatbot is expected to behave
BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user) "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!",
)

# This is the message with which the system opens the conversation
WELCOME_MSG = "Welcome to the BaristaBot cafe. Type 'q' to quit. How may I serve you today?"

print("State and system instructions defined!")

In [ ]:
# API Key Setup
# If you're running locally, set GOOGLE_API_KEY environment variable with your API key
# Get your API key from: https://ai.google.dev/
# If running in Kaggle, the API key should be set from the secrets

if not os.getenv("GOOGLE_API_KEY"):
    # Try to get from Kaggle secrets
    try:
        from kaggle_secrets import UserSecretsClient
        GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
        os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
        print("API key loaded from Kaggle secrets")
    except:
        print("Warning: GOOGLE_API_KEY not set. Please set it in your environment.")
else:
    print("API key already set in environment")

# Initialize the Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-latest")
print("LLM initialized successfully!")

In [ ]:
import os
from typing import Annotated, Literal
from typing_extensions import TypedDict
from collections.abc import Iterable
from random import randint
from pprint import pprint

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.messages.tool import ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, InjectedState
from IPython.display import Image

print("All imports successful!")

In [ ]:
%pip install -qU "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0" "langchain-core" "pillow"
print("Dependency installation completed successfully!")

# BaristaBot: Building a Cafe Ordering System with LangGraph and Gemini API

This notebook demonstrates how to create a stateful application using LangGraph that integrates with the Gemini API to build an interactive cafe ordering system called **BaristaBot**.

## Learning Objectives
- Create stateful applications using LangGraph
- Integrate Gemini API (via LangChain) into LangGraph applications
- Define and manipulate state using TypedDict
- Simulate dynamic, tool-augmented behavior with menus and ordering
- Model conditional transitions and loops for user interaction
- Handle tool calls using LangGraph's ToolNode mechanism

## What We'll Build
A conversational cafe ordering system (BaristaBot) that:
- Takes coffee/tea orders using natural language
- Offers a real-time menu via tools
- Confirms and modifies orders
- Loops through conversation until an order is placed
- Handles tool calls using LangGraph's ToolNode